In [1]:
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [ ]:
df = pd.read_csv('../data/sentiment_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12744 entries, 0 to 12743
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   content                 12744 non-null  object
 1   label                   12744 non-null  object
 2   type                    12744 non-null  object
 3   content_without_emojis  12744 non-null  object
 4   content_preprocessed    12744 non-null  object
dtypes: object(5)
memory usage: 497.9+ KB


In [ ]:
df['label'].value_counts()

label
neg    6372
pos    6372
Name: count, dtype: int64

In [ ]:
df.head()

,content,label,type,content_without_emojis,content_preprocessed
0,les draps commandés - très onéreux - sont mal ...,neg,commentaire,les draps commandés - très onéreux - sont mal ...,"['drap', 'commandé', 'onéreux', 'mal', 'ajuste..."
1,impossible de les joindre suite à une commande...,neg,commentaire,impossible de les joindre suite à une commande...,"['impossible', 'joindre', 'suite', 'commande',..."
2,dommage qu'on ne puisse pas noter rapidement l...,neg,commentaire,dommage qu'on ne puisse pas noter rapidement l...,"['dommage', 'quon', 'pouvoir', 'noter', 'rapid..."
3,"7,80 pour une crêpe sans nutella +une piteuse ...",neg,commentaire,"7,80 pour une crêpe sans nutella +une piteuse ...","['780', 'crêpe', 'nuteller', 'piteux', 'boul',..."
4,calendrier de l’avent commandé le 25/11 livrai...,neg,commentaire,calendrier de l’avent commandé le 25/11 livrai...,"['calendrier', 'aver', 'commander', '2511', 'l..."


In [ ]:
# df["content_without_emojis"].replace(":", "", regex=True, inplace=True)
# df["content_without_emojis"].replace("_", "", regex=True, inplace=True)
import ast

df['content_preprocessed'] = df['content_preprocessed'].apply(ast.literal_eval)

In [ ]:
X= df['content']
y = df['label']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
stopwords =  stopwords.words('french')


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(min_df=2,
                             max_df=0.8,
                             sublinear_tf=True,
                             ngram_range=(1,3),
                             strip_accents='unicode',
                             use_idf=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:
df['content_preprocessed'].apply(lambda x: ' '.join(x))

0        drap commandé onéreux mal ajuster trop grand d...
1        impossible joindre suite commande avoir questi...
2        dommage quon pouvoir noter rapidement aliment ...
3        780 crêpe nuteller piteux boul glace trouver s...
4        calendrier aver commander 2511 livraison midéc...
                               ...                        
12739    commande non recevoir bout dun mois livraison ...
12740                   bel arnaqu jaurai devoir lire avis
12741    vraie blagu boîte chocolat non livrer problème...
12742                excellent bon accueil rapide 2 mois j
12743    sandwich dégoûter cétait ok cest goût thon bai...
Name: content_preprocessed, Length: 12744, dtype: object

In [ ]:
df.head()

,content,label,type,content_without_emojis,content_preprocessed
0,les draps commandés - très onéreux - sont mal ...,neg,commentaire,les draps commandés - très onéreux - sont mal ...,"[drap, commandé, onéreux, mal, ajuster, trop, ..."
1,impossible de les joindre suite à une commande...,neg,commentaire,impossible de les joindre suite à une commande...,"[impossible, joindre, suite, commande, avoir, ..."
2,dommage qu'on ne puisse pas noter rapidement l...,neg,commentaire,dommage qu'on ne puisse pas noter rapidement l...,"[dommage, quon, pouvoir, noter, rapidement, al..."
3,"7,80 pour une crêpe sans nutella +une piteuse ...",neg,commentaire,"7,80 pour une crêpe sans nutella +une piteuse ...","[780, crêpe, nuteller, piteux, boul, glace, tr..."
4,calendrier de l’avent commandé le 25/11 livrai...,neg,commentaire,calendrier de l’avent commandé le 25/11 livrai...,"[calendrier, aver, commander, 2511, livraison,..."


In [ ]:
trains_vectors = vectorizer.fit_transform(X_train)
test_vectors = vectorizer.transform(X_test)

In [ ]:
clf_svm_ovo = svm.SVC(decision_function_shape='ovo')
clf_svm_ovo.fit(trains_vectors, y_train)
clf_svm_ovo_pred = clf_svm_ovo.predict(test_vectors)
report = classification_report(y_test, clf_svm_ovo_pred, output_dict=True)
accuracy_score_svm_ovo = accuracy_score(y_test, clf_svm_ovo_pred)
print("positive:", report['pos'])
print("negative:", report['neg'])
print("accuracy:", accuracy_score_svm_ovo)

positive: {'precision': 0.9644308943089431, 'recall': 0.9052464228934817, 'f1-score': 0.9339019189765458, 'support': 3145.0}
negative: {'precision': 0.9128654970760234, 'recall': 0.9674620390455532, 'f1-score': 0.9393711448773883, 'support': 3227.0}
accuracy: 0.9367545511613308


In [ ]:
clf_svm_ovr = svm.SVC(decision_function_shape='ovr')
clf_svm_ovr.fit(trains_vectors, y_train)
clf_svm_ovr_pred = clf_svm_ovr.predict(test_vectors)
report = classification_report(y_test, clf_svm_ovr_pred, output_dict=True)
accuracy_score_svm_ovr = accuracy_score(y_test, clf_svm_ovr_pred)
print("positive:", report['pos'])
print("negative:", report['neg'])
# print("neutral:", report['neutre'])
print("accuracy:", accuracy_score_svm_ovr)

In [ ]:
model_ovo = svm.LinearSVC(multi_class='crammer_singer')
model_ovo.fit(trains_vectors, y_train)
model_pred = model_ovo.predict(test_vectors)
report = classification_report(y_test, model_pred, output_dict=True)
accuracy_score_model_ovo = accuracy_score(y_test, model_pred)
print("positive:", report['pos'])
print("negative:", report['neg'])
# print("neutral:", report['neutre'])
print("accuracy:", accuracy_score_model_ovo)

positive: {'precision': 0.9549638395792241, 'recall': 0.9236883942766295, 'f1-score': 0.9390657830935834, 'support': 3145.0}
negative: {'precision': 0.9279279279279279, 'recall': 0.9575457080880074, 'f1-score': 0.9425041939911545, 'support': 3227.0}
accuracy: 0.9408349026993095


In [ ]:
model_ovr = svm.LinearSVC(multi_class='ovr')
model_ovr.fit(trains_vectors, y_train)
model_pred = model_ovr.predict(test_vectors)
report = classification_report(y_test, model_pred, output_dict=True)
accuracy_score_model_ovr = accuracy_score(y_test, model_pred)
print("positive:", report['pos'])
print("negative:", report['neg'])
# print("neutral:", report['neutre'])
print("accuracy:", accuracy_score_model_ovr)

positive: {'precision': 0.9544253632760898, 'recall': 0.918918918918919, 'f1-score': 0.9363356552729629, 'support': 3145.0}
negative: {'precision': 0.923744019138756, 'recall': 0.9572358227455842, 'f1-score': 0.9401917516359762, 'support': 3227.0}
accuracy: 0.9383239171374764


In [ ]:
import time

classifier_linear = svm.SVC(kernel='linear')
t0 = time.time()
classifier_linear.fit(trains_vectors, y_train)
t1 = time.time()
prediction_linear = classifier_linear.predict(test_vectors)
t2 = time.time()
time_linear_train = t1 - t0
time_linear_predict = t2 - t1

print(f"Training time (linear kernel): {time_linear_train:.3f} seconds")
print(f"Prediction time (linear kernel): {time_linear_predict:.3f} seconds")

report = classification_report(y_test, prediction_linear, output_dict=True)
accuracy_score_linear = accuracy_score(y_test, prediction_linear)
print("positive:", report['pos'])
print("negative:", report['neg'])
# print("neutral:", report['neutre'])
print('accuracy_score:', accuracy_score_linear)

Training time (linear kernel): 24.007 seconds
Prediction time (linear kernel): 20.589 seconds
positive: {'precision': 0.9608363757052771, 'recall': 0.9205087440381559, 'f1-score': 0.9402403377720039, 'support': 3145.0}
negative: {'precision': 0.9255730872283418, 'recall': 0.9634335295940502, 'f1-score': 0.944123899180079, 'support': 3227.0}
accuracy_score: 0.9422473320778405


In [ ]:
nlp = spacy.load('fr_core_news_sm')

In [ ]:
import emoji

In [ ]:
def preprocess_text(text):
    text = emoji.demojize(text, language="fr").replace(':', " ").replace('_',' ')
    doc = nlp(text)    
    text = [
        token.lemma_
        for token in doc
        if not token.is_punct
        and not token.is_space
    ]
    return [word for word in text if word not in stopwords]
    

In [ ]:
review01 = """Jeune humain assidu"""
review_vector = vectorizer.transform([review01])
classifier_linear.predict(review_vector)

array(['pos'], dtype=object)

In [ ]:
review01 = """visage_souriant_avec_yeux_en_forme_de_coeur"""
review_vector = vectorizer.transform([review01])
classifier_linear.predict(review_vector)

array(['pos'], dtype=object)

In [ ]:
review01 = """visage sourire yeux coeur"""
review_vector = vectorizer.transform([review01])
classifier_linear.predict(review_vector)

array(['pos'], dtype=object)

In [ ]:
from sklearn import naive_bayes
from sklearn.metrics import f1_score, precision_score, recall_score


clfrNB = naive_bayes.MultinomialNB()
clfrNB.fit(trains_vectors, y_train)
predicted_labels = clfrNB.predict(test_vectors)
accuracy_score_nb = accuracy_score(y_test, predicted_labels)
precision_nb = precision_score(y_test, predicted_labels, average='micro')
recall_nb = recall_score(y_test, predicted_labels, average='micro')
print("accuracy:", accuracy_score_nb)
print("precision:", precision_nb)
print("recall:", recall_nb)
print("f1_score:", f1_score(y_test, predicted_labels, average='micro'))
print(classification_report(y_test, predicted_labels, output_dict=True))

accuracy: 0.9278091650973007
precision: 0.9278091650973007
recall: 0.9278091650973007
f1_score: 0.9278091650973007
{'neg': {'precision': 0.891815349759275, 'recall': 0.9758289432909824, 'f1-score': 0.9319325244155076, 'support': 3227.0}, 'pos': {'precision': 0.9725448785638859, 'recall': 0.8785373608903021, 'f1-score': 0.9231540260608085, 'support': 3145.0}, 'accuracy': 0.9278091650973007, 'macro avg': {'precision': 0.9321801141615804, 'recall': 0.9271831520906422, 'f1-score': 0.927543275238158, 'support': 6372.0}, 'weighted avg': {'precision': 0.9316606680408979, 'recall': 0.9278091650973007, 'f1-score': 0.9275997596123802, 'support': 6372.0}}


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

clLR = LogisticRegression(random_state=42)
clLR.fit(trains_vectors, y_train)
predicted_labels = clLR.predict(test_vectors)
accuracy_score_lr = accuracy_score(y_test, predicted_labels)
precision_lr = precision_score(y_test, predicted_labels, average='micro')
recall_lr = recall_score(y_test, predicted_labels, average='micro')
print("accuracy:", accuracy_score_lr)
print("precision:", precision_lr)
print("recall:", recall_lr)
print("f1_score:", f1_score(y_test, predicted_labels, average='micro'))

accuracy: 0.9265536723163842
precision: 0.9265536723163842
recall: 0.9265536723163842
f1_score: 0.9265536723163842


In [ ]:

review01 = """Salut ça va? Moi je vais bien"""
# text = emoji.demojize(review01, language='fr').replace(':', ' ').replace('_', ' ')
review_vector = vectorizer.transform([review01])
# print(pd.date_)
print(" ".join(preprocess_text(review01)))
print(classifier_linear.predict(review_vector))
print(model_ovo.predict(review_vector))
print(model_ovr.predict(review_vector))
print(clf_svm_ovo.predict(review_vector))
print(clf_svm_ovr.predict(review_vector))
print(clfrNB.predict(review_vector))
print(clLR.predict(review_vector))

Salut cela aller Moi aller bien
['pos']
['pos']
['pos']
['pos']
['pos']
['neg']
['pos']


In [ ]:
review02="""Bof Beaucoup d'idées mais aucune explication sur la mise en place"""
review_vector = vectorizer.transform([" ".join(preprocess_text(review02))])
print(classifier_linear.predict(review_vector))
print(model_ovo.predict(review_vector))
print(model_ovr.predict(review_vector))
print(clf_svm_ovo.predict(review_vector))
print(clf_svm_ovr.predict(review_vector))
print(clfrNB.predict(review_vector))
print(clLR.predict(review_vector))

['neg']
['neg']
['neg']
['neg']
['neg']
['neg']
['neg']


In [ ]:
review02="""Je ne pense pas que ce projet puisse être considéré comme une réussite, car malgré les nombreuses réunions, les promesses répétées, les ressources investies et le temps consacré à son développement, aucun des objectifs principaux n'a été atteint, les délais n'ont pas été respectés, les utilisateurs ne sont pas satisfaits du résultat final, les problèmes techniques continuent de s'accumuler et aucune solution durable n'a encore été mise en place pour corriger les difficultés observées depuis le début."""
review_vector = vectorizer.transform([" ".join(preprocess_text(review02))])
print(' '.join(preprocess_text(review02)))
print(classifier_linear.predict(review_vector))
print(model_ovo.predict(review_vector))
print(model_ovr.predict(review_vector))
print(clf_svm_ovo.predict(review_vector))
print(clf_svm_ovr.predict(review_vector))
print(clfrNB.predict(review_vector))
print(clLR.predict(review_vector))

penser projet pouvoir être considérer comme réussite car malgré nombreux réunion promesse répéter ressource investir temps consacrer développement aucun objectif principal avoir être atteindre délai avoir être respecter utilisateur être satisfaire résultat final problème technique continuer accumuler aucun solution durable avoir encore être mettre place corriger difficulté observer depuis début
['neg']
['neg']
['neg']
['neg']
['neg']
['neg']
['neg']


In [ ]:
review04="""je pleure """

review_vector = vectorizer.transform([" ".join(preprocess_text(review04))])
print(" ".join(preprocess_text(review04)))
print(classifier_linear.predict(review_vector))
print(model_ovo.predict(review_vector))
print(model_ovr.predict(review_vector))
print(clf_svm_ovo.predict(review_vector))
print(clf_svm_ovr.predict(review_vector))
print(clfrNB.predict(review_vector))
print(clLR.predict(review_vector))

pleurer
['pos']
['pos']
['pos']
['pos']
['pos']
['pos']
['pos']


In [ ]:
review03="""À deux reprises, des mauvaises expériences..
J'ai commandé quelques articles à hauteur de 204920 frs, mais fût ma déception de voir, en lieu et place d'un sac au dos pour l'école, on me livre un box à boîte goûter, cette boîte qui coûte à peine 3000 frs, m'a été facturé au prix du sac au dos, c'est à dire 10900frs.
Livré par une certaine Aicha Madjid.
Il n'y a aucun contrôle, c'est vraiment de l'anarchie.
Plus jamais sur Jumia !😡 😡"""
review_vector = vectorizer.transform([" ".join(preprocess_text(review03))])
print(" ".join(preprocess_text(review03)))
print(classifier_linear.predict(review_vector))
print(model_ovr.predict(review_vector))
print(model_ovo.predict(review_vector))
print(clf_svm_ovo.predict(review_vector))
print(clf_svm_ovr.predict(review_vector))
print(clfrNB.predict(review_vector))
print(clLR.predict(review_vector))

deux reprise mauvais expérience avoir commander quelque article hauteur 204920 fr déception voir lieu place sac dos école livre box boîte goûter boîte coûter peine 3000 fr avoir être facturer prix sac dos être dire 10900frs Livré certaine Aicha Madjid avoir aucun contrôle être vraiment anarchie plus jamais Jumia visage_boudeur visage_boudeur
['neg']
['neg']
['neg']
['neg']
['neg']
['neg']
['neg']


In [ ]:
# Create a comparison dataframe for all accuracy scores
accuracy_comparison = pd.DataFrame({
    'Classifier': ['SVM OVO', 'SVM OVR', 'LinearSVC OVO', 'LinearSVC OVR', 'SVC Linear', 'Naive Bayes', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score_svm_ovo, 
        accuracy_score_svm_ovr, 
        accuracy_score_model_ovr,
        accuracy_score_model_ovo,
        accuracy_score_linear,
        accuracy_score_nb,
        accuracy_score_lr,
    ]
})

print("\n=== Accuracy Score Comparison ===")
print(accuracy_comparison.to_string(index=False))
print(f"\nBest Model: {accuracy_comparison.loc[accuracy_comparison['Accuracy'].idxmax(), 'Classifier']}")
print(f"Best Accuracy: {accuracy_comparison['Accuracy'].max():.4f}")


=== Accuracy Score Comparison ===
         Classifier  Accuracy
            SVM OVO  0.936755
            SVM OVR  0.936755
      LinearSVC OVO  0.938324
      LinearSVC OVR  0.940835
         SVC Linear  0.942247
        Naive Bayes  0.927809
Logistic Regression  0.926554

Best Model: SVC Linear
Best Accuracy: 0.9422


In [ ]:
# import pickle
# filename="classifier_linear_sentiment.pkl"
# with open(filename, 'wb') as file:
#     pickle.dump(classifier_linear, file)

In [ ]:
# with open("tfidf_vectorizer.pkl", "wb") as file:
#     pickle.dump(vectorizer, file)